# Module 09: Pandas — Solutions

**Dataset:** Titanic  
**Objective:** Complete solutions for all exercises

In [ ]:
import pandas as pd
import numpy as np

url = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
titanic = pd.read_csv(url)
print('Titanic dataset loaded. Shape:', titanic.shape)

### Solution 1: Data Inspection

In [ ]:
print('First 8 rows:')
print(titanic.head(8))
print('\nColumn data types:')
print(titanic.dtypes)
print('\nSummary statistics:')
print(titanic.describe())
print('\nUnique values per column:')
print(titanic.nunique())

### Solution 2: Boolean Indexing

In [ ]:
print('Female survivors:', len(titanic.loc[(titanic['Sex'] == 'female') & (titanic['Survived'] == 1)]))
print('Male under 18, not survived:', len(titanic.loc[(titanic['Sex'] == 'male') & (titanic['Age'] < 18) & (titanic['Survived'] == 0)]))
fare_75 = titanic['Fare'].quantile(0.75)
print('High fare passengers (fare > 75th percentile):', len(titanic.loc[titanic['Fare'] > fare_75]))
print('1st class, age > 60:', len(titanic.loc[(titanic['Pclass'] == 1) & (titanic['Age'] > 60)]))

### Solution 3: Missing Data Handling

In [ ]:
print('Missing % per column:')
print((titanic.isna().sum() / len(titanic) * 100).round(2))

titanic['Age'] = titanic.groupby('Pclass')['Age'].transform(lambda x: x.fillna(x.median()))
titanic.drop(columns=['Cabin'], inplace=True)
titanic.dropna(subset=['Embarked'], inplace=True)

print('\nMissing values remaining:', titanic.isna().sum().sum())

### Solution 4: GroupBy Operations

In [ ]:
print('Survival rate by Pclass and Sex:')
print(titanic.groupby(['Pclass', 'Sex'])['Survived'].mean().round(3))

print('\nAvg age and fare by Pclass and Embarked:')
print(titanic.groupby(['Pclass', 'Embarked'])[['Age', 'Fare']].mean().round(2))

print('\nMax fare per Pclass:')
print(titanic.loc[titanic.groupby('Pclass')['Fare'].idxmax()][['Pclass', 'Name', 'Fare']])

titanic['Fare_deviation'] = titanic.groupby('Pclass')['Fare'].transform(lambda x: x - x.mean())
print('\nFare deviation from Pclass mean (head):')
print(titanic[['Pclass', 'Fare', 'Fare_deviation']].head())

### Solution 5: Apply & Map

In [ ]:
titanic['Pclass_Name'] = titanic['Pclass'].map({1: 'First', 2: 'Second', 3: 'Third'})
print('Pclass mapped:', titanic['Pclass_Name'].unique())

titanic['Has_Cabin'] = titanic['Cabin'].notna() if 'Cabin' in titanic.columns else False
# Recreate since we dropped Cabin
orig = pd.read_csv(url)
titanic['Has_Cabin'] = orig['Cabin'].notna()
print('Has_Cabin value counts:', titanic['Has_Cabin'].value_counts().to_dict())

titanic['Total_Family'] = titanic.apply(lambda row: row['SibSp'] + row['Parch'], axis=1)

def age_bin(age):
    if pd.isna(age):
        return 'Unknown'
    if age < 13:
        return 'Child'
    if age < 20:
        return 'Teen'
    if age < 40:
        return 'Adult'
    if age < 60:
        return 'Middle'
    return 'Senior'

titanic['Age_Bin'] = titanic['Age'].apply(age_bin)
print('Age bins:', titanic['Age_Bin'].value_counts().to_dict())

### Solution 6: String Operations

In [ ]:
titanic['Title'] = titanic['Name'].str.extract(r',\s*([^\.]+)\.', expand=False)
print('Title distribution:', titanic['Title'].value_counts().to_dict())

mrs_miss_mask = titanic['Name'].str.contains('Mrs|Miss', na=False)
print('Names containing Mrs or Miss:', mrs_miss_mask.sum())

names_start_A = titanic['Name'].str.startswith('A')
print('Names starting with A:', names_start_A.sum())

titanic['Surname'] = titanic['Name'].str.split(',').str[0]
print('\nSurname preview:', titanic['Surname'].head().tolist())

### Solution 7: Merge & Pivot Tables

In [ ]:
deck_info = pd.DataFrame({
    'Deck': list('ABCDEFG'),
    'Deck_Level': list(range(1, 8))
})
print('Deck info table:')
print(deck_info)

titanic['Deck'] = orig['Cabin'].str.extract(r'([A-Z])', expand=False)
merged = pd.merge(titanic, deck_info, on='Deck', how='left')
print('\nAfter merge with deck_info:')
print(merged[['Name', 'Deck', 'Deck_Level']].dropna().head())

pivot = pd.pivot_table(titanic, values='Fare', index='Pclass', columns='Embarked', aggfunc='mean')
print('\nPivot table - Avg Fare by Pclass and Embarked:')
print(pivot.round(2))

melted = pd.melt(pivot.reset_index(), id_vars='Pclass', value_vars=['C', 'Q', 'S'],
                 var_name='Embarked', value_name='Avg_Fare')
print('\nMelted pivot table:')
print(melted)

### Solution 8: Memory Optimization

In [ ]:
orig_memory = orig.memory_usage(deep=True).sum()
print('Original memory:', orig_memory, 'bytes')

opt = orig.copy()
for col in opt.select_dtypes(include='object').columns:
    if opt[col].nunique() < 10:
        opt[col] = opt[col].astype('category')

for col in opt.select_dtypes(include='float').columns:
    opt[col] = pd.to_numeric(opt[col], downcast='float')

opt_memory = opt.memory_usage(deep=True).sum()
print('Optimized memory:', opt_memory, 'bytes')
print('Savings:', orig_memory - opt_memory, 'bytes ({:.1%})'.format(1 - opt_memory/orig_memory))

### Solution 9: Multi-Index

In [ ]:
mi = titanic.set_index(['Pclass', 'Sex', 'Survived']).sort_index()
print('Multi-index levels:', mi.index.names)

female_survivors = mi.xs((1, 'female'), level=('Survived', 'Sex'))
print('\nFemale survivors (xs):')
print(female_survivors[['Age', 'Fare']].head())

class1 = mi.xs(1, level='Pclass')
print('\nClass 1 passengers:', len(class1))

print('\nAverage Age by multi-index:')
print(mi.groupby(level=['Pclass', 'Sex', 'Survived'])['Age'].mean().round(1))

### Solution 10: ML Data Preparation Pipeline

In [ ]:
# Full pipeline
raw = pd.read_csv(url)

# 1. Impute Age
raw['Age'] = raw.groupby(['Pclass', 'Sex'])['Age'].transform(lambda x: x.fillna(x.median()))

# 2. Feature engineering
raw['Title'] = raw['Name'].str.extract(r',\s*([^\.]+)\.', expand=False)
raw['Family_Size'] = raw['SibSp'] + raw['Parch'] + 1
raw['Is_Alone'] = (raw['Family_Size'] == 1).astype(int)

# 3. Fill remaining missing
raw['Embarked'].fillna(raw['Embarked'].mode()[0], inplace=True)
raw['Fare'].fillna(raw['Fare'].median(), inplace=True)

# 4. Create features for ML
features = raw[['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked', 'Title', 'Family_Size', 'Is_Alone']].copy()
y = raw['Survived']

# 5. Encode categoricals
features = pd.get_dummies(features, columns=['Sex', 'Embarked', 'Title'], drop_first=True, dtype=int)

print('X shape:', features.shape)
print('X columns:', features.columns.tolist())
print('Missing values:', features.isna().sum().sum())
print('\nFeature matrix (first 5 rows):')
print(features.head())
print('\ny distribution:\n', y.value_counts(normalize=True).round(3))